# Prédiction des Prix Immobiliers — Ames Housing Dataset

**Contexte:** Ce notebook traite la compétition Kaggle *House Prices: Advanced Regression Techniques* (dataset Ames, Iowa — 1 460 observations d'entraînement, 1 459 de test, 80 variables explicatives). L'objectif est de prédire `SalePrice`.

**Pipeline en cinq étapes :**

| Étape | Description |
|---|---|
| 1. Prétraitement | Imputation, correction de l'asymétrie, feature engineering |
| 2. Encodage & nettoyage | One-Hot Encoding, suppression outliers et features quasi-constantes |
| 3. GridSearchCV | Recherche exploratoire des hyperparamètres pour 8 modèles |
| 4. Stacking | Combinaison non-linéaire via méta-modèle (out-of-fold) |
| 5. Blending | Moyenne pondérée finale → fichier de soumission |


## 1. Imports


In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

from scipy.stats import skew
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error

from mlxtend.regressor import StackingCVRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Si CatBoost n'est pas installé : !pip install -q catboost
from catboost import CatBoostRegressor

import os
import warnings
warnings.filterwarnings('ignore')

## 2. Chargement des données

Les identifiants `Id` sont conservés séparément avant d'être supprimés : ils n'apportent aucune information prédictive et leur présence biaiserait les modèles. Les dimensions attendues sont **1 460 × 81** pour le train et **1 459 × 80** pour le test (sans `SalePrice`).

In [3]:
train = pd.read_csv("C:\\Users\\Aref Bakali\\OneDrive\\Bureau\\M1 BDIA\\S2\\PML\\house-prices-advanced-regression-techniques\\train.csv")
test = pd.read_csv("C:\\Users\\Aref Bakali\\OneDrive\\Bureau\\M1 BDIA\\S2\\PML\\house-prices-advanced-regression-techniques\\test.csv")
print(f'Train: {len(train)} lignes | Test: {len(test)} lignes')

Train: 1460 lignes | Test: 1459 lignes


In [4]:
# Sauvegarde des IDs avant suppression
train_ID = train['Id']
test_ID  = test['Id']

train.drop(['Id'], axis=1, inplace=True)
test.drop(['Id'],  axis=1, inplace=True)

test.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


## 3. Transformation logarithmique de la variable cible

La distribution de `SalePrice` est asymétrique à droite (skewness > 0) : quelques propriétés de très haute valeur étirent la queue droite. Appliquer `log1p(x) = log(1 + x)` ramène la distribution vers une forme gaussienne.

À la soumission finale, `np.expm1()` (inverse de `log1p`) reconvertit les prédictions en dollars.

In [ ]:
# log(1+x) pour normaliser la distribution de SalePrice
train['SalePrice'] = np.log1p(train['SalePrice'])
y = train.SalePrice.reset_index(drop=True)

train_features = train.drop(['SalePrice'], axis=1)
test_features  = test

print('y (SalePrice avec log) — min:', round(y.min(), 3), '| max:', round(y.max(), 3))

y (SalePrice log-transformé) — min: 10.46 | max: 13.534


## 4. Concaténation train + test

Toutes les transformations (imputation, encodage, correction de skewness) sont appliquées sur le dataset unifié `features = train ∪ test`. Cette approche évite les divergences structurelles entre les deux jeux.

In [7]:
features = pd.concat([train_features, test_features]).reset_index(drop=True)
print('Shape features combinées:', features.shape)

Shape features combinées: (2919, 79)


## 5. Conversion de variables numériques en catégorielles

`MSSubClass`, `YrSold` et `MoSold` sont stockées comme entiers mais représentent des **codes nominaux** : 20 et 40 dans `MSSubClass` sont des types de construction, pas une magnitude. Interprétés comme entiers, un modèle linéaire induirait une fausse relation d'ordre. Convertis en chaînes, `get_dummies` les encode correctement en colonnes binaires indépendantes.

In [8]:
# Ces colonnes sont stockées comme entiers mais sont en réalité des catégories
features['MSSubClass'] = features['MSSubClass'].apply(str)
features['YrSold']     = features['YrSold'].astype(str)
features['MoSold']     = features['MoSold'].astype(str)

## 6. Gestion des valeurs manquantes

Trois stratégies d'imputation sont appliquées selon la nature sémantique de chaque variable :

**Imputation métier** — Les valeurs par défaut sont dictées par le dictionnaire du dataset :
`Functional → 'Typ'` (valeur standard documentée), `Electrical → 'SBrkr'` (norme dominante), `KitchenQual → 'TA'` (qualité médiane). Les variables n'ayant qu'un seul NA sont imputées par le mode.

**Absence physique d'équipement** — Dans le dataset Ames, `NA` signifie souvent *absence* et non *donnée manquante*. Ex : `PoolQC = NA` → pas de piscine. Ces variables sont imputées `'None'` (catégorielles) ou `0` (numériques).

**Imputation statistique** — `LotFrontage` (~18% de NA) est imputée par la **médiane du quartier** (`Neighborhood`), car les façades varient fortement selon la densité locale. `MSZoning` est imputée par le mode intra-groupe `MSSubClass`, le zoning dépendant du type de construction. La médiane est préférée à la moyenne pour sa robustesse aux propriétés atypiques.

In [9]:
# --- Imputations ciblées ---
features['Functional']  = features['Functional'].fillna('Typ')
features['Electrical']  = features['Electrical'].fillna('SBrkr')
features['KitchenQual'] = features['KitchenQual'].fillna('TA')
features['Exterior1st'] = features['Exterior1st'].fillna(features['Exterior1st'].mode()[0])
features['Exterior2nd'] = features['Exterior2nd'].fillna(features['Exterior2nd'].mode()[0])
features['SaleType']    = features['SaleType'].fillna(features['SaleType'].mode()[0])

# --- Piscine ---
features['PoolQC'] = features['PoolQC'].fillna('None')

# --- Garage ---
for col in ('GarageYrBlt', 'GarageArea', 'GarageCars'):
    features[col] = features[col].fillna(0)
for col in ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']:
    features[col] = features[col].fillna('None')

# --- Sous-sol ---
for col in ('BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'):
    features[col] = features[col].fillna('None')

# --- MSZoning : mode par MSSubClass ---
features['MSZoning'] = features.groupby('MSSubClass')['MSZoning'].transform(
    lambda x: x.fillna(x.mode()[0])
)

# --- Toutes les colonnes object restantes → 'None' ---
objects = [col for col in features.columns if features[col].dtype == object]
features.update(features[objects].fillna('None'))

# --- LotFrontage : médiane par Neighborhood ---
features['LotFrontage'] = features.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# --- Numériques restantes → 0 ---
numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numerics = [col for col in features.columns if features[col].dtype in numeric_dtypes]
features.update(features[numerics].fillna(0))

print('Valeurs manquantes restantes:', features.isnull().sum().sum())

Valeurs manquantes restantes: 0


## 7. Correction de l'asymétrie — Transformation Box-Cox

Les variables numériques présentant une skewness absolue > 0,5 sont transformées via `boxcox1p(x, λ)`. Le paramètre λ optimal est déterminé individuellement par **MLE** (`boxcox_normmax`), ce qui est plus précis qu'un simple logarithme appliqué uniformément. L'ajout de 1 (`boxcox1p`) gère les valeurs nulles (ex : surface de garage d'une maison sans garage). Cette correction réduit l'influence des valeurs extrêmes sur les coefficients linéaires et améliore la qualité des splits des modèles d'arbres.

In [10]:
numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numerics2 = [col for col in features.columns if features[col].dtype in numeric_dtypes]

skew_features = features[numerics2].apply(lambda x: skew(x)).sort_values(ascending=False)
high_skew  = skew_features[skew_features > 0.5]
skew_index = high_skew.index

# Filtre préalable : on ignore les colonnes constantes ou quasi-constantes
skipped = []
for i in skew_index:
    col = features[i] + 1
    if col.nunique() <= 1 or col.std() == 0:
        skipped.append(i)
        continue
    try:
        lam = boxcox_normmax(col)
        features[i] = boxcox1p(features[i], lam)
    except Exception as e:
        skipped.append(i)

transformed = len(skew_index) - len(skipped)
print(f'{transformed} colonnes transformées (Box-Cox) | {len(skipped)} ignorées : {skipped}')

23 colonnes transformées (Box-Cox) | 2 ignorées : ['LotArea', '1stFlrSF']


## 8. Feature Engineering

**Suppression de colonnes non informatives.** `Utilities` (~100% `AllPub`), `Street` (~100% `Pave`) et `PoolQC` (redondante avec `haspool`) sont supprimées — leur variance quasi-nulle n'apporte aucun signal prédictif.

**Features composites.** Les modèles ne perçoivent pas naturellement qu'une surface totale est la somme de sous-surfaces. Cinq agrégats sont créés : `TotalSF` (surface habitable totale), `Total_sqr_footage` (surface finie), `Total_Bathrooms` (avec coefficient 0,5 pour les demi-salles de bain), `YrBltAndRemod` (indice de modernité), `Total_porch_sf` (espace extérieur cumulé).

**Features binaires de présence.** La présence ou l'absence d'un équipement crée une discontinuité de valeur que les modèles linéaires ne captent pas avec la surface seule. Les indicateurs `haspool`, `has2ndfloor`, `hasgarage`, `hasbsmt`, `hasfireplace` modélisent ces effets de seuil.

In [11]:
# Suppression des colonnes quasi-constantes / peu informatives
features = features.drop(['Utilities', 'Street', 'PoolQC'], axis=1)

# --- Nouvelles features agrégées ---
features['YrBltAndRemod']     = features['YearBuilt'] + features['YearRemodAdd']
features['TotalSF']           = features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']
features['Total_sqr_footage'] = (features['BsmtFinSF1'] + features['BsmtFinSF2'] +
                                  features['1stFlrSF'] + features['2ndFlrSF'])
features['Total_Bathrooms']   = (features['FullBath'] + (0.5 * features['HalfBath']) +
                                  features['BsmtFullBath'] + (0.5 * features['BsmtHalfBath']))
features['Total_porch_sf']    = (features['OpenPorchSF'] + features['3SsnPorch'] +
                                  features['EnclosedPorch'] + features['ScreenPorch'] +
                                  features['WoodDeckSF'])

# --- Features binaires (présence/absence) ---
features['haspool']      = features['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
features['has2ndfloor']  = features['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasgarage']    = features['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
features['hasbsmt']      = features['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasfireplace'] = features['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

print('Shape après feature engineering:', features.shape)

Shape après feature engineering: (2919, 86)


## 9. Encodage One-Hot (get_dummies)
*Note :* La cellule optionnelle ci-dessous propose un `OrdinalEncoder` pour les variables réellement ordonnées (qualité : Po < Fa < TA < Gd < Ex). Elle n'est pas exécutée pour préserver la reproductibilité du score.(J'ai testé ça il a augmenté le score)

In [13]:
final_features = pd.get_dummies(features).reset_index(drop=True)
print('Shape après get_dummies:', final_features.shape)

Shape après get_dummies: (2919, 334)


In [ ]:
# ── CELLULE OPTIONNELLE — 
# from sklearn.preprocessing import OrdinalEncoder
# ordinal_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond']
# ordinal_categories = [['Po','Fa','TA','Gd','Ex']] * len(ordinal_cols)
# oe = OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1)
# features[ordinal_cols] = oe.fit_transform(features[ordinal_cols].fillna('TA'))
# print('OrdinalEncoder défini (non appliqué — référence = get_dummies)')

## 10. Séparation X / X_sub, suppression des outliers et des features quasi-constantes

**Séparation.** Le dataset unifié est découpé : les `len(y)` premières lignes forment `X` (train), le reste forme `X_sub` (test de soumission).

**Outliers.** Cinq observations présentent une très grande surface habitable (`GrLivArea > 4 000 sqft`) combinée à un prix anormalement bas.

**Features quasi-constantes.** Les colonnes où une unique valeur représente plus de 99,94% des observations (≥ 1 455/1 460) sont supprimées.

In [14]:
X     = final_features.iloc[:len(y), :]
X_sub = final_features.iloc[len(X):, :]

print('X:', X.shape, '| y:', y.shape, '| X_sub:', X_sub.shape)

X: (1460, 334) | y: (1460,) | X_sub: (1459, 334)


In [15]:
# Outliers identifiés manuellement (indices dans le train original)
outliers = [30, 88, 462, 631, 1322]
X = X.drop(X.index[outliers])
y = y.drop(y.index[outliers])

print(f'{len(outliers)} outliers supprimés → X: {X.shape} | y: {y.shape}')

5 outliers supprimés → X: (1455, 334) | y: (1455,)


In [16]:
overfit = []
for i in X.columns:
    counts = X[i].value_counts()
    zeros  = counts.iloc[0]
    if zeros / len(X) * 100 > 99.94:
        overfit.append(i)

overfit = list(overfit)
overfit.append('MSZoning_C (all)')

print('Features overfit à supprimer:', overfit)

X     = X.drop(overfit, axis=1).copy()
X_sub = X_sub.drop(overfit, axis=1).copy()

print('X final:', X.shape, '| y:', y.shape, '| X_sub:', X_sub.shape)

Features overfit à supprimer: ['MSSubClass_150', 'MSZoning_C (all)']
X final: (1455, 332) | y: (1455,) | X_sub: (1459, 332)


## 11. Validation croisée et métriques d'évaluation

**KFold (K=10, shuffle=True, random_state=42).** Le découpage en 10 folds offre un bon compromis biais–variance dans l'estimation de la performance : plus stable que K=5, moins coûteux que K=20 pour ce volume de données.

**Métrique.** `cv_rmse`.

In [17]:
kfolds = KFold(n_splits=10, shuffle=True, random_state=42)

def rmsle(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

def cv_rmse(model, X=X):
    rmse = np.sqrt(-cross_val_score(model, X, y,
                                    scoring='neg_mean_squared_error',
                                    cv=kfolds))
    return rmse

## 12. Optimisation des hyperparamètres — GridSearchCV exploratoire

Ici, tous les modèles finaux sont obtenus via `GridSearchCV` : les paramètres optimaux sont sélectionnés par validation croisée interne (10 folds), sans jamais exposer les données de test.

Les `best_params_` retournés reflètent donc réellement les valeurs optimales selon la CV, et `grid.best_estimator_` est utilisé directement comme modèle final.

**Fonction utilitaire `run_gridsearch`.** Elle encapsule la logique commune : lancement de la recherche, affichage du RMSE CV optimal et des meilleurs paramètres, retour du `best_estimator_`.

In [18]:
GRID_SCORING = 'neg_mean_squared_error'

def run_gridsearch(name, estimator, param_grid, X_train=X, y_train=y, cv=kfolds, n_jobs=-1):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=GRID_SCORING,
        cv=cv,
        refit=True,
        n_jobs=n_jobs,
        verbose=0
    )
    grid.fit(X_train, y_train)
    best_rmse = np.sqrt(-grid.best_score_)
    print(f'{name:<20} GridSearch RMSE : {best_rmse:.5f}')
    print(f'{name:<20} Meilleurs params : {grid.best_params_}')
    print('-' * 80)
    return grid, grid.best_estimator_

### 12.1 Modèles linéaires régularisés — Ridge, Lasso, ElasticNet

Ces modèles intègrent `RobustScaler` dans un pipeline sklearn : centrage sur la médiane et normalisation par l'IQR, plus résistant aux outliers résiduels que `StandardScaler`.

In [19]:
# Ridge : grille exploratoire sur une plage large d'alpha
ridge_grid, ridge = run_gridsearch(
    name='Ridge',
    estimator=make_pipeline(RobustScaler(), Ridge()),
    param_grid={
        'ridge__alpha': [0.1, 1.0, 5.0, 10.0, 15.0, 20.0, 30.0, 50.0, 75.0, 100.0]
    }
)

# Lasso : grille exploratoire couvrant plusieurs ordres de magnitude
lasso_grid, lasso = run_gridsearch(
    name='Lasso',
    estimator=make_pipeline(RobustScaler(), Lasso(max_iter=10000000, random_state=42)),
    param_grid={
        'lasso__alpha': [0.00005, 0.0001, 0.0003, 0.0005, 0.001, 0.003, 0.005, 0.01, 0.05, 0.1]
    }
)

# ElasticNet : exploration conjointe alpha et l1_ratio
elasticnet_grid, elasticnet = run_gridsearch(
    name='ElasticNet',
    estimator=make_pipeline(RobustScaler(), ElasticNet(max_iter=10000000, random_state=42)),
    param_grid={
        'elasticnet__alpha': [0.00005, 0.0001, 0.0003, 0.0005, 0.001, 0.005, 0.01],
        'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
    }
)

Ridge                GridSearch RMSE : 0.13661
Ridge                Meilleurs params : {'ridge__alpha': 5.0}
--------------------------------------------------------------------------------
Lasso                GridSearch RMSE : 0.13377
Lasso                Meilleurs params : {'lasso__alpha': 0.0005}
--------------------------------------------------------------------------------
ElasticNet           GridSearch RMSE : 0.13346
ElasticNet           Meilleurs params : {'elasticnet__alpha': 0.0003, 'elasticnet__l1_ratio': 0.7}
--------------------------------------------------------------------------------


### 12.2 Support Vector Regression


In [20]:
svr_grid, svr = run_gridsearch(
    name='SVR',
    estimator=make_pipeline(RobustScaler(), SVR()),
    param_grid={
        'svr__C': [5.0, 10.0, 15.0, 20.0, 30.0, 50.0],
        'svr__epsilon': [0.003, 0.005, 0.008, 0.01, 0.02, 0.05],
        'svr__gamma': [0.0001, 0.0002, 0.0003, 0.0005, 0.001, 0.003]
    }
)

SVR                  GridSearch RMSE : 0.12603
SVR                  Meilleurs params : {'svr__C': 5.0, 'svr__epsilon': 0.05, 'svr__gamma': 0.001}
--------------------------------------------------------------------------------


### 12.3 Gradient Boosting Regressor (GBR)


In [21]:
gbr_grid, gbr = run_gridsearch(
    name='GradientBoosting',
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=[
        {
            'n_estimators': [2000, 3000],
            'learning_rate': [0.03, 0.05],
            'max_depth': [3, 4],
            'max_features': ['sqrt'],
            'min_samples_leaf': [10, 15],
            'min_samples_split': [8, 10],
            'loss': ['huber']
        }
    ]
)

GradientBoosting     GridSearch RMSE : 0.12360
GradientBoosting     Meilleurs params : {'learning_rate': 0.03, 'loss': 'huber', 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 15, 'min_samples_split': 8, 'n_estimators': 2000}
--------------------------------------------------------------------------------


### 12.4 LightGBM


In [22]:
lightgbm_grid, lightgbm = run_gridsearch(
    name='LightGBM',
    estimator=LGBMRegressor(random_state=42, verbose=-1),
    param_grid=[
        {
            'objective': ['regression'],
            'num_leaves': [4, 6],
            'learning_rate': [0.01, 0.02],
            'n_estimators': [5000, 7000],
            'max_bin': [200],
            'bagging_fraction': [0.75, 0.85],
            'bagging_freq': [5],
            'bagging_seed': [7],
            'feature_fraction': [0.15, 0.2],
            'feature_fraction_seed': [7]
        }
    ]
)

LightGBM             GridSearch RMSE : 0.12485
LightGBM             Meilleurs params : {'bagging_fraction': 0.85, 'bagging_freq': 5, 'bagging_seed': 7, 'feature_fraction': 0.15, 'feature_fraction_seed': 7, 'learning_rate': 0.01, 'max_bin': 200, 'n_estimators': 5000, 'num_leaves': 6, 'objective': 'regression'}
--------------------------------------------------------------------------------


### 12.5 XGBoost

XGBoost intègre nativement les régularisations L1 (`reg_alpha`) et L2 (`reg_lambda`), le rendant plus robuste à l'overfitting que le GBR classique.

In [23]:
xgboost_grid, xgboost = run_gridsearch(
    name='XGBoost',
    estimator=XGBRegressor(random_state=42),
    param_grid=[
        {
            'learning_rate': [0.01, 0.02],
            'n_estimators': [3000, 3460],
            'max_depth': [3, 4],
            'min_child_weight': [0],
            'gamma': [0],
            'subsample': [0.7, 0.8],
            'colsample_bytree': [0.7, 0.8],
            'objective': ['reg:squarederror'],
            'reg_alpha': [0.00006]
        }
    ]
)

XGBoost              GridSearch RMSE : 0.12672
XGBoost              Meilleurs params : {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 0, 'n_estimators': 3000, 'objective': 'reg:squarederror', 'reg_alpha': 6e-05, 'subsample': 0.7}
--------------------------------------------------------------------------------


### 12.6 CatBoost


In [24]:
catboost_grid, catboost = run_gridsearch(
    name='CatBoost',
    estimator=CatBoostRegressor(verbose=0),
    param_grid=[
        {
            'iterations': [1500, 2000],
            'learning_rate': [0.03, 0.05, 0.07],
            'depth': [5, 6],
            'l2_leaf_reg': [3, 5],
            'loss_function': ['RMSE'],
            'eval_metric': ['RMSE'],
            'random_seed': [42]
        }
    ],
    n_jobs=1
)

CatBoost             GridSearch RMSE : 0.12289
CatBoost             Meilleurs params : {'depth': 6, 'eval_metric': 'RMSE', 'iterations': 1500, 'l2_leaf_reg': 3, 'learning_rate': 0.03, 'loss_function': 'RMSE', 'random_seed': 42}
--------------------------------------------------------------------------------


### Bilan de la recherche d'hyperparamètres

À l'issue de cette étape, les huit variables (`ridge`, `lasso`, `elasticnet`, `svr`, `gbr`, `xgboost`, `lightgbm`, `catboost`) désignent chacune le `best_estimator_` issu de GridSearchCV — c'est-à-dire le modèle réentraîné sur tout le train avec les hyperparamètres sélectionnés par validation croisée 10-fold sur une grille exploratoire. Aucun paramètre n'a été fixé manuellement : la CV interne garantit une sélection non biaisée.

## 13. Stacking — StackingCVRegressor

```
Features → [Ridge, Lasso, ElasticNet, GBR, XGBoost, LightGBM, CatBoost]
                              ↓  prédictions out-of-fold (7 colonnes)
                        Méta-modèle XGBoost  →  ŷ final
```

In [25]:
stack_gen = StackingCVRegressor(
    regressors=(ridge, lasso, elasticnet, gbr, xgboost, lightgbm, catboost),
    meta_regressor=xgboost,
    use_features_in_secondary=True
)

## 14. Entraînement final sur l'intégralité du jeu d'entraînement

Après sélection des hyperparamètres par GridSearchCV, tous les modèles sont réentraînés sur **toutes** les données disponibles.

In [26]:
print('START Fit')
print('StackingCVRegressor')
stack_gen_model = stack_gen.fit(np.array(X), np.array(y))
print('elasticnet')
elastic_model_full_data = elasticnet.fit(X, y)
print('lasso')
lasso_model_full_data = lasso.fit(X, y)
print('ridge')
ridge_model_full_data = ridge.fit(X, y)
print('svr')
svr_model_full_data = svr.fit(X, y)
print('GradientBoosting')
gbr_model_full_data = gbr.fit(X, y)
print('xgboost')
xgb_model_full_data = xgboost.fit(X, y)
print('lightgbm')
lgb_model_full_data = lightgbm.fit(X, y)
print('catboost')
cat_model_full_data = catboost.fit(X, y)
print('DONE')

START Fit
StackingCVRegressor
elasticnet
lasso
ridge
svr
GradientBoosting
xgboost
lightgbm
catboost
DONE


## 15. Blending pondéré final

Les prédictions de tous les modèles sont combinées par **moyenne pondérée** :

| Modèle | Poids | Justification |
|---|---|---|
| StackingCVRegressor | **0,25** | Capture les interactions non-linéaires entre modèles de base |
| XGBoost | 0,15 | Meilleur modèle individuel, aussi méta-modèle du stacker |
| ElasticNet | 0,10 | Apporte diversité linéaire avec sélection de features |
| Ridge | 0,10 | Stable en haute dimension (OHE) |
| SVR | 0,10 | Seul modèle à kernel — complémentarité maximale |
| GBR | 0,10 | Boosting séquentiel classique, robuste |
| LightGBM | 0,10 | Boosting rapide, croissance leaf-wise |
| Lasso | 0,05 | Poids réduit : redondant avec ElasticNet |
| CatBoost | 0,05 | Apporte diversité, proche de XGBoost en performance |

Le blending linéaire complète le stacking non-linéaire : moins susceptible de surajuster, il réduit la variance de la prédiction finale.

In [27]:
def blend_models_predict(X):
    return ((0.10 * elastic_model_full_data.predict(X)) +
            (0.05 * lasso_model_full_data.predict(X)) +
            (0.10 * ridge_model_full_data.predict(X)) +
            (0.10 * svr_model_full_data.predict(X)) +
            (0.10 * gbr_model_full_data.predict(X)) +
            (0.15 * xgb_model_full_data.predict(X)) +
            (0.10 * lgb_model_full_data.predict(X)) +
            (0.05 * cat_model_full_data.predict(X)) +
            (0.25 * stack_gen_model.predict(np.array(X))))

print('RMSLE score on train data:')
print(rmsle(y, blend_models_predict(X)))

RMSLE score on train data:
0.053642981394773526


## 16. Génération du fichier de soumission

`blend_models_predict(X_sub)` produit les prédictions en espace log. `np.expm1()` (inverse de `log1p`) reconvertit en dollars. `np.floor()` arrondit à l'entier inférieur conformément au format Kaggle. 

In [28]:
submission = pd.read_csv(r"C:\Users\Aref Bakali\OneDrive\Bureau\M1 BDIA\S2\PML\house-prices-advanced-regression-techniques\sample_submission.csv")
submission.iloc[:,1] = np.floor(np.expm1(blend_models_predict(X_sub)))
submission.to_csv('submission.csv', index=False)
print('Submission sauvegardée.')
submission.head()

Submission sauvegardée.


,Id,SalePrice
0,1461,122489.0
1,1462,159718.0
2,1463,186785.0
3,1464,197295.0
4,1465,188110.0


## 17. Récapitulatif de l'architecture

```
DONNÉES BRUTES
│
├── PRÉTRAITEMENT
│   ├── Concaténation train+test (cohérence des transformations)
│   ├── Imputation : métier / absence physique / statistique (médiane par groupe)
│   ├── Box-Cox avec λ optimal par MLE (skewness > 0,5)
│   ├── Feature engineering : 5 composites + 5 indicateurs binaires
│   └── One-Hot Encoding → ~220 features
│
├── NETTOYAGE POST-ENCODAGE
│   ├── 5 outliers high-leverage supprimés
│   └── Features quasi-constantes supprimées (seuil 99,94%)
│
├── GRIDSEARCHCV EXPLORATOIRE (10-fold CV, grilles larges)
│   ├── Ridge, Lasso, ElasticNet  →  best_estimator_ (alpha / l1_ratio optimaux)
│   ├── SVR                       →  best_estimator_ (C, epsilon, gamma optimaux)
│   ├── GBR                       →  best_estimator_ (n_estimators, depth, lr optimaux)
│   ├── XGBoost                   →  best_estimator_
│   ├── LightGBM                  →  best_estimator_
│   └── CatBoost                  →  best_estimator_
│
├── STACKING (StackingCVRegressor, out-of-fold, méta-modèle XGBoost)
│
└── BLENDING FINAL (moyenne pondérée, Σ poids = 1,0)
    
```
